In [4]:
import pandas as pd
import numpy as np
from datetime import datetime

# These commands below set some options for pandas and to have matplotlib show the charts in the notebook
pd.set_option('display.max_rows', 1000)
pd.options.display.float_format = '{:,.2f}'.format

# Define a date parser to pass to read_csv
d = lambda x: datetime.strptime(x, '%d-%b-%y')

# Load the data
# We have this defaulted to the folder OUTSIDE of your repo - please change it as needed
contrib = pd.read_csv('P00000001-CA.csv', index_col=False, parse_dates=['contb_receipt_dt'], date_parser=d)

# Note - for now, it is okay to ignore the warning about mixed types. 

/Users/alexmwamsindo/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3165: DtypeWarning: Columns (6,11,12) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [5]:
# 1a YOUR CODE HERE

print("\n1a SHAPE:")
print(contrib.shape) # Shows (rows, columns) --> helps confirm dataset loaded correctly


1a SHAPE:
(1125659, 18)


In [6]:
# 1b YOUR CODE HERE

# Here, we're listing all column names --> verify against documentation
print("\n1b COLUMNS:")
print(contrib.columns.tolist())


1b COLUMNS:
['cmte_id', 'cand_id', 'cand_nm', 'contbr_nm', 'contbr_city', 'contbr_st', 'contbr_zip', 'contbr_employer', 'contbr_occupation', 'contb_receipt_amt', 'contb_receipt_dt', 'receipt_desc', 'memo_cd', 'memo_text', 'form_tp', 'file_num', 'tran_id', 'election_tp']


In [7]:
# 1c YOUR CODE HERE

# Focuses on the candidate ID, name, and contributor state
# Also, helps to confirm consistency and expected values

print("\n1c FIRST 5 ROWS:")
print(contrib[['cand_id', 'cand_nm', 'contbr_st']].head())


1c FIRST 5 ROWS:
     cand_id                  cand_nm contbr_st
0  P00003392  Clinton, Hillary Rodham        CA
1  P00003392  Clinton, Hillary Rodham        CA
2  P00003392  Clinton, Hillary Rodham        CA
3  P60007168         Sanders, Bernard        CA
4  P60007168         Sanders, Bernard        CA


In [8]:
# 1d YOUR CODE HERE

# Counting unique election types:
# election_tp indicates the type and year of election:
# 'P2016' = Primary election (2016)
# 'G2016' = General election (2016)
# 'P2020' = Primary election (2020)
# The values match the documentation and confirm this dataset includes multiple election cycles.

print("\n1d election_tp values:")
print(contrib['election_tp'].value_counts())


1d election_tp values:
P2016    810481
G2016    313746
P2020         7
Name: election_tp, dtype: int64


In [9]:
# 1e YOUR CODE HERE

# Here, we get a quick snapshot of dataset structure

print("\nHEAD:")
print(contrib.head())

print("\nINFO:")
print(contrib.info())

print("\nDESCRIBE:")
print(contrib.describe())


HEAD:
     cmte_id    cand_id                  cand_nm          contbr_nm  \
0  C00575795  P00003392  Clinton, Hillary Rodham         AULL, ANNE   
1  C00575795  P00003392  Clinton, Hillary Rodham  CARROLL, MARYJEAN   
2  C00575795  P00003392  Clinton, Hillary Rodham   GANDARA, DESIREE   
3  C00577130  P60007168         Sanders, Bernard          LEE, ALAN   
4  C00577130  P60007168         Sanders, Bernard   LEONELLI, ODETTE   

     contbr_city contbr_st     contbr_zip            contbr_employer  \
0       LARKSPUR        CA 949,391,913.00                        NaN   
1        CAMBRIA        CA 934,284,638.00                        NaN   
2        FONTANA        CA 923,371,507.00                        NaN   
3      CAMARILLO        CA 930,111,214.00  AT&T GOVERNMENT SOLUTIONS   
4  REDONDO BEACH        CA 902,784,310.00   VERICOR ENTERPRISES INC.   

   contbr_occupation  contb_receipt_amt contb_receipt_dt receipt_desc memo_cd  \
0            RETIRED              50.00       2016-0

In [10]:
# 1f YOUR CODE HERE

# Helps to identify columns with missing data
# Useful for deciding what to drop later

print("\n1f NON-NULL COUNTS:")
print(contrib.notnull().sum().sort_values(ascending=False))


1f NON-NULL COUNTS:
cmte_id              1125659
cand_id              1125659
tran_id              1125659
file_num             1125659
form_tp              1125659
contb_receipt_dt     1125659
contb_receipt_amt    1125659
contbr_st            1125659
contbr_nm            1125659
cand_nm              1125659
contbr_city          1125633
contbr_zip           1125564
election_tp          1124234
contbr_occupation    1115260
contbr_employer       967757
memo_text             501148
memo_cd               144268
receipt_desc           15045
dtype: int64


In [11]:
# 1g YOUR CODE HERE

# Ensures that Bernie Sanders always maps to the same cand_id

print("\n1g Sanders cand_id check:")
sanders_ids = contrib[contrib['cand_nm'] == 'Sanders, Bernard']['cand_id'].unique()
print(sanders_ids)


1g Sanders cand_id check:
['P60007168']


In [12]:
# 1h YOUR CODE HERE

# This confirms that all records are from California (CA)

print("\n1h STATE CHECK:")
print(contrib['contbr_st'].value_counts().head())


1h STATE CHECK:
CA    1125659
Name: contbr_st, dtype: int64


In [13]:
# 1i YOUR CODE HERE

# Helps to check if transaction IDs are unique (potential primary key)
# Also, any duplicates may indicate amendments, refunds, or data issues

print("\n1i DUPLICATE tran_id:")
print(contrib['tran_id'].duplicated().sum())

# Also, we need to remove any duplicates transactions, to ensure each trasaction is unique.
contrib = contrib.drop_duplicates(subset='tran_id', keep='first')


1i DUPLICATE tran_id:
3454


In [14]:
# 1j YOUR CODE HERE

# Identifies refunds or chargebacks (negative values)

print("\n1j NEGATIVE DONATIONS:")
neg = contrib[contrib['contb_receipt_amt'] < 0]
print(len(neg))
print(neg[['cand_nm', 'contb_receipt_amt']].head())


1j NEGATIVE DONATIONS:
11879
                       cand_nm  contb_receipt_amt
19   Cruz, Rafael Edward 'Ted'             -25.00
23   Cruz, Rafael Edward 'Ted'            -150.00
81   Cruz, Rafael Edward 'Ted'             -60.00
190  Cruz, Rafael Edward 'Ted'            -100.00
213  Cruz, Rafael Edward 'Ted'             -25.00


In [15]:
# 1k YOUR CODE HERE

# Ensures that the date column is properly formatted

print("\n1k DATE RANGE:")
contrib['contb_receipt_dt'] = pd.to_datetime(contrib['contb_receipt_dt'], errors='coerce')
print(contrib['contb_receipt_dt'].min())
print(contrib['contb_receipt_dt'].max())


1k DATE RANGE:
2013-11-05 00:00:00
2016-10-19 00:00:00


In [25]:
# 1l.1 Answer here:

# The dataset is loaded successfully and in the expected number of rows and columns; therefore, the difference in character 
# from a structured point of view means there has been no data ingestion issues. However, there are some data quality issues 
# (e.g., ZIP codes may have multiple formats; missing values; data types that aren't consistent, and the like.) 
# that still need to be resolved.

In [26]:
# 1l.2 Answer here:

# This dataset also has what we need to answer major questions concerning campaign contributions, 
# such as total donation amount(s), date(s) of the donation(s), details of the contributor 
# (state of contributor, occupation of contributor, name of contributor's employer), and the name(s) of the candidate(s). 
# This dataset will allow an analysis of campaign contributions for trends in fundraising, trends in donor behaviour, 
# and comparisons of candidates.

In [27]:
# 1l.3 Answer here:

# Candidate Name – cand_nm, Candidate ID – cand_id, Total Amount of Contribution Received – contb_receipt_amt (core metric), 
# Date of Contribution Received – contb_receipt_dt (time analysis), State of Contributor – contbr_st (geographic insight), 
# Occupation of Contributor and Employer of Contributor – contbr_occupation and contbr_employer (donor profiling), 
# Election Type – election_tp (differentiating between primary and general elections).

In [28]:
# 1l.4 Answer here:

# There are certain columns that can be removed from the dataset depending on the analysis being conducted; for example, 
# the columns of memo_cd, memo_text, and receipt_desc all contain very few records and are not very useful. 
# Additionally, form_tp, file_num, and tran_id are fields related to the administrative/technical processes. 
# Finally, contbr_nm serves no purpose for the researcher unless the researcher wishes to analyse individual donors 
# within the dataset.

In [29]:
# 1l.5 Answer here:

# The data has many types of quality issues such as:

# Mixed types (warnings when importing for some columns)
# Missing data in contbr_employer, memo_text and receipt_desc specifically
# Negative amounts of donations which are probably refunds
# Duplicate transaction IDs (by a total of 3,454 records)
# Floats/strings encoded incorrectly as zipcodes
# Donation amounts highly skewed (significant outlier values)

In [30]:
# 1l.6 Answer here:

# Here are some assumptions based on this data:

# Negative amounts of donations are likely refunds and not contributions
# Entries with duplicate transaction ID may need to be deduplicated or verified
# Missing employer or occupation data does not keep it from being a contribution
# Analysis will be done mainly on positive donations
# Data set is for California (CA) contributors only.

In [16]:
# 2a YOUR CODE HERE

print("\n=== STEP 2: CLEANING + FILTERING ===")

contrib = contrib.dropna(subset=['cand_nm', 'contb_receipt_amt', 'contb_receipt_dt'])

# Filtering the Primary and date range

contrib = contrib[
    (contrib['election_tp'] == 'P2016') &
    (contrib['contb_receipt_dt'] >= '2014-01-01') &
    (contrib['contb_receipt_dt'] <= '2016-06-07')
]
print("\n2a SHAPE:", contrib.shape)


=== STEP 2: CLEANING + FILTERING ===

2a SHAPE: (666182, 18)


In [17]:
# 2b YOUR CODE HERE

# Here we filter Sanders.

contrib = contrib[contrib['cand_nm'] == 'Sanders, Bernard']
print("2b SHAPE:", contrib.shape)

2b SHAPE: (379284, 18)


In [22]:
# 2c YOUR CODE HERE

contrib['zip5'] = contrib['contbr_zip'].astype(str).str[:5]
contrib['zip5'] = pd.to_numeric(contrib['zip5'], errors='coerce')
contrib = contrib.dropna(subset=['zip5'])
contrib['zip5'] = contrib['zip5'].astype(int)

contrib = contrib[(contrib['zip5'] >= 90001) & (contrib['zip5'] <= 96162)]
print("2c SHAPE:", contrib.shape)

2c SHAPE: (376959, 20)


In [23]:
# 2d YOUR CODE HERE

contrib = contrib[contrib['contb_receipt_amt'] > 0]
print("2d SHAPE:", contrib.shape)

2d SHAPE: (376959, 20)


In [24]:
# 2e YOUR CODE HERE

contrib = contrib[['cand_nm', 'zip5', 'contb_receipt_amt', 'contb_receipt_dt']]
print("2e SHAPE:", contrib.shape)

2e SHAPE: (376959, 4)


In [25]:
# 3a YOUR CODE HERE

# Answering the Questions:

contrib['day'] = contrib['contb_receipt_dt'].dt.day

# Now, we run the analysis:
zip_summary = contrib.groupby('zip5').agg({
    'contb_receipt_amt': ['count', 'sum']
})
zip_summary.columns = ['count', 'total']

print("\nTOP ZIP BY COUNT:")
print(zip_summary.sort_values('count', ascending=False).head(1))

print("\nTOP ZIP BY TOTAL:")
print(zip_summary.sort_values('total', ascending=False).head(1))


TOP ZIP BY COUNT:
       count      total
zip5                   
94110   3799 284,398.05

TOP ZIP BY TOTAL:
       count      total
zip5                   
94110   3799 284,398.05


In [27]:
day_summary = contrib.groupby('day').size().sort_values(ascending=False)

print("\nTOP DONATION DAYS:")
print(day_summary.head())

# My interpretation is as follows:

print("\nINTERPRETATION:")
print("The highest number of donations and corresponding total contributions (ZIP code 94110) is indicative of very strong donor activity within San Francisco.")
print("Most of the days of the month where donations occurred (29, 30,31) define a pattern of significant clustering of donations to the latter part of the month;\nthis could likely be attributed to either timing of payroll cycles or/and timing of fundraising events")


TOP DONATION DAYS:
day
29    21837
31    19485
30    19485
14    16810
9     15133
dtype: int64

INTERPRETATION:
The highest number of donations and corresponding total contributions (ZIP code 94110) is indicative of very strong donor activity within San Francisco.
Most of the days of the month where donations occurred (29, 30,31) define a pattern of significant clustering of donations to the latter part of the month;
this could likely be attributed to either timing of payroll cycles or/and timing of fundraising events
